In [ ]:
# V5 Cell 1 — Admissibility Envelope Setup

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

print("=" * 60)
print("V5 — SERVE ADMISSIBILITY ENVELOPE")
print("=" * 60)

print("\nGoal:")
print("  Map the set of launch conditions that")
print("  produce a legal serve.")

print("\nFree launch variables:")
print("  θ = elevation angle")
print("  φ = azimuth angle")

print("\nFixed baseline conditions:")
print("  Speed = 200 km/h")
print("  Contact height = 3.0 m")
print("  Spin = 0 rad/s")

print("\nConstraints:")
print("  1. Clear the net")
print("  2. Land beyond the net")
print("  3. Land before the service line")
print("  4. Land inside the target service box")

print("\nV5 SETUP: READY")

In [ ]:
# V5 Cell 2 — Serve Physics + Court Model

# -------------------------------------------------------
# Physical constants
# -------------------------------------------------------

G = 9.81

BALL_MASS = 0.0575
BALL_DIAMETER = 0.067
BALL_RADIUS = BALL_DIAMETER / 2
BALL_AREA = np.pi * BALL_RADIUS**2

AIR_DENSITY = 1.21
DRAG_COEFFICIENT = 0.55

# -------------------------------------------------------
# Court geometry
# -------------------------------------------------------

COURT_WIDTH = 8.23
SERVICE_BOX_WIDTH = COURT_WIDTH / 2

NET_X = 0.0
SERVICE_LINE_X = 6.40
BASELINE_X = 11.885

NET_HEIGHT_CENTER = 0.914
NET_HEIGHT_POST = 1.07

SERVER_X = -BASELINE_X
CONTACT_HEIGHT = 3.0


# -------------------------------------------------------
# Aerodynamic model
# -------------------------------------------------------

def lift_coefficient(speed, spin_speed):
    """
    Provisional lift-coefficient model.
    """

    if speed <= 0 or spin_speed <= 0:
        return 0.0

    return 1.0 / (
        2.0 + speed / spin_speed
    )


def magnus_acceleration(
    velocity,
    omega
):
    """
    Calculate Magnus acceleration.
    """

    velocity = np.asarray(
        velocity,
        dtype=float
    )

    omega = np.asarray(
        omega,
        dtype=float
    )

    speed = np.linalg.norm(
        velocity
    )

    spin_rate = np.linalg.norm(
        omega
    )

    if speed == 0 or spin_rate == 0:
        return np.zeros(3)

    velocity_hat = (
        velocity / speed
    )

    omega_hat = (
        omega / spin_rate
    )

    spin_speed = (
        BALL_RADIUS * spin_rate
    )

    C_L = lift_coefficient(
        speed,
        spin_speed
    )

    magnus_direction = np.cross(
        omega_hat,
        velocity_hat
    )

    force_magnitude = (
        0.5
        * AIR_DENSITY
        * BALL_AREA
        * C_L
        * speed**2
    )

    return (
        force_magnitude
        * magnus_direction
        / BALL_MASS
    )


def serve_acceleration(
    velocity,
    omega
):
    """
    Total acceleration:
    gravity + drag + Magnus.
    """

    velocity = np.asarray(
        velocity,
        dtype=float
    )

    speed = np.linalg.norm(
        velocity
    )

    # Gravity
    a_gravity = np.array([
        0.0,
        0.0,
        -G
    ])

    # Drag
    if speed > 0:

        drag_factor = (
            -0.5
            * AIR_DENSITY
            * DRAG_COEFFICIENT
            * BALL_AREA
            * speed
            / BALL_MASS
        )

        a_drag = (
            drag_factor
            * velocity
        )

    else:

        a_drag = np.zeros(3)

    # Magnus
    a_magnus = magnus_acceleration(
        velocity,
        omega
    )

    return (
        a_gravity
        + a_drag
        + a_magnus
    )


# -------------------------------------------------------
# Trajectory equation
# -------------------------------------------------------

def serve_trajectory(
    t,
    state,
    omega
):

    x, y, z, vx, vy, vz = state

    velocity = np.array([
        vx,
        vy,
        vz
    ])

    acceleration = serve_acceleration(
        velocity,
        omega
    )

    return np.array([
        vx,
        vy,
        vz,
        acceleration[0],
        acceleration[1],
        acceleration[2]
    ])


# -------------------------------------------------------
# Court events
# -------------------------------------------------------

def net_event(t, state):

    return state[0] - NET_X


net_event.terminal = False
net_event.direction = 1


def landing_event(t, state):

    return state[2]


landing_event.terminal = True
landing_event.direction = -1


# -------------------------------------------------------
# Serve simulation
# -------------------------------------------------------

def simulate_serve(
    speed_kmh,
    launch_angle_deg,
    azimuth_deg,
    omega=np.zeros(3)
):
    """
    Simulate one serve.
    """

    speed = speed_kmh / 3.6

    theta = np.radians(
        launch_angle_deg
    )

    phi = np.radians(
        azimuth_deg
    )

    vx = (
        speed
        * np.cos(theta)
        * np.cos(phi)
    )

    vy = (
        speed
        * np.cos(theta)
        * np.sin(phi)
    )

    vz = (
        speed
        * np.sin(theta)
    )

    initial_state = np.array([
        SERVER_X,
        0.0,
        CONTACT_HEIGHT,
        vx,
        vy,
        vz
    ])

    solution = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),

        t_span=(0.0, 4.0),

        y0=initial_state,

        events=[
            net_event,
            landing_event
        ],

        rtol=1e-10,
        atol=1e-12,
        max_step=0.001,

        dense_output=True
    )

    return solution


# -------------------------------------------------------
# Court constraints
# -------------------------------------------------------

def net_height(y):

    return (
        NET_HEIGHT_CENTER
        + (
            NET_HEIGHT_POST
            - NET_HEIGHT_CENTER
        )
        * np.minimum(
            np.abs(y) / SERVICE_BOX_WIDTH,
            1.0
        )
    )


def net_clearance(
    z,
    y
):

    return (
        z - net_height(y)
    )


def is_inside_service_box(
    y,
    target_side="deuce"
):

    if target_side == "deuce":

        return (
            0.0
            <= y
            <= SERVICE_BOX_WIDTH
        )

    elif target_side == "ad":

        return (
            -SERVICE_BOX_WIDTH
            <= y
            <= 0.0
        )

    else:

        raise ValueError(
            "target_side must be "
            "'deuce' or 'ad'"
        )


def extract_serve_events(
    solution
):

    if len(solution.t_events[0]) == 0:

        raise RuntimeError(
            "Serve did not cross the net."
        )

    if len(solution.t_events[1]) == 0:

        raise RuntimeError(
            "Serve did not reach the court."
        )

    net_state = (
        solution.y_events[0][0]
    )

    landing_state = (
        solution.y_events[1][0]
    )

    return (
        net_state,
        landing_state
    )


print("=" * 60)
print("V5 — VERIFIED SERVE MODEL LOADED")
print("=" * 60)

print("\nPhysics:")
print("  Gravity + quadratic drag + Magnus")

print("\nCourt:")
print(f"  Net:            x = {NET_X:.3f} m")
print(f"  Service line:   x = {SERVICE_LINE_X:.3f} m")
print(f"  Service width:  {SERVICE_BOX_WIDTH:.3f} m")

print("\nLaunch:")
print(f"  Server x:       {SERVER_X:.3f} m")
print(f"  Contact height: {CONTACT_HEIGHT:.3f} m")

print("\nModel limitation:")
print("  Magnus coefficient remains provisional.")

print("\nV5 SERVE MODEL: READY")

In [ ]:
# V5 Cell 3 — Regression Test Against V4

# Known V4 test case
test_speed = 200.0
test_angle = -7.914436
test_azimuth = 0.0
test_spin = np.zeros(3)

# Expected V4 values
expected_landing_x = 5.282063
expected_net_clearance = 0.162464

# Run the same simulation in V5
solution = simulate_serve(
    speed_kmh=test_speed,
    launch_angle_deg=test_angle,
    azimuth_deg=test_azimuth,
    omega=test_spin
)

net_state, landing_state = extract_serve_events(
    solution
)

# Calculate V5 values
actual_landing_x = landing_state[0]

actual_net_clearance = net_clearance(
    net_state[2],
    net_state[1]
)

# Differences
landing_error = abs(
    actual_landing_x
    - expected_landing_x
)

clearance_error = abs(
    actual_net_clearance
    - expected_net_clearance
)


print("=" * 60)
print("V5 — REGRESSION TEST AGAINST V4")
print("=" * 60)

print("\nTest parameters")
print("-" * 60)
print(f"Speed:          {test_speed:.3f} km/h")
print(f"Launch angle:   {test_angle:.6f}°")
print(f"Azimuth:        {test_azimuth:.3f}°")
print("Spin:           0 rad/s")

print("\nLanding distance")
print("-" * 60)
print(f"V4 reference:   {expected_landing_x:.6f} m")
print(f"V5 result:      {actual_landing_x:.6f} m")
print(f"Absolute error: {landing_error:.3e} m")

print("\nNet clearance")
print("-" * 60)
print(f"V4 reference:   {expected_net_clearance:.6f} m")
print(f"V5 result:      {actual_net_clearance:.6f} m")
print(f"Absolute error: {clearance_error:.3e} m")

# Verification tolerance
landing_pass = landing_error < 1e-5
clearance_pass = clearance_error < 1e-5

print("\nVerification")
print("-" * 60)

print(
    f"Landing distance regression: "
    f"{'PASS' if landing_pass else 'FAIL'}"
)

print(
    f"Net clearance regression: "
    f"{'PASS' if clearance_pass else 'FAIL'}"
)

if landing_pass and clearance_pass:
    print("\nV5 REGRESSION TEST: PASS")
else:
    print("\nV5 REGRESSION TEST: FAIL")

In [ ]:
# V5 Cell 4 — Complete Legal-Serve Classifier

def classify_serve(
    speed_kmh,
    launch_angle_deg,
    azimuth_deg,
    omega=np.zeros(3),
    target_side="deuce"
):
    """
    Simulate and classify a tennis serve.

    Returns a dictionary containing the trajectory
    measurements and individual constraint results.
    """

    # Simulate trajectory
    solution = simulate_serve(
        speed_kmh=speed_kmh,
        launch_angle_deg=launch_angle_deg,
        azimuth_deg=azimuth_deg,
        omega=omega
    )

    # Extract net and landing states
    net_state, landing_state = (
        extract_serve_events(solution)
    )

    # Net measurements
    net_x = net_state[0]
    net_y = net_state[1]
    net_z = net_state[2]

    clearance = net_clearance(
        net_z,
        net_y
    )

    # Landing measurements
    landing_x = landing_state[0]
    landing_y = landing_state[1]
    landing_z = landing_state[2]

    # Individual constraints
    clears_net = (
        clearance > 0.0
    )

    inside_depth = (
        NET_X < landing_x < SERVICE_LINE_X
    )

    inside_width = (
        is_inside_service_box(
            landing_y,
            target_side=target_side
        )
    )

    legal = (
        clears_net
        and inside_depth
        and inside_width
    )

    return {
        "speed_kmh": speed_kmh,
        "launch_angle_deg": launch_angle_deg,
        "azimuth_deg": azimuth_deg,

        "net_x": net_x,
        "net_y": net_y,
        "net_z": net_z,
        "net_clearance": clearance,

        "landing_x": landing_x,
        "landing_y": landing_y,
        "landing_z": landing_z,

        "clears_net": clears_net,
        "inside_depth": inside_depth,
        "inside_width": inside_width,

        "legal": legal
    }


print("=" * 60)
print("V5 — LEGAL-SERVE CLASSIFIER")
print("=" * 60)

print("\nClassifier outputs:")
print("  • Net clearance")
print("  • Landing position")
print("  • Net constraint")
print("  • Depth constraint")
print("  • Width constraint")
print("  • Final legal/fault classification")

print("\nV5 CLASSIFIER: DEFINED")

In [ ]:
# V5 Cell 5 — Legal-Serve Classifier Verification

test_cases = [
    {
        "name": "Legal — Deuce",
        "speed": 200.0,
        "angle": -7.914436,
        "azimuth": 0.0,
        "side": "deuce",
        "expected": True
    },
    {
        "name": "Legal — Ad",
        "speed": 200.0,
        "angle": -7.914436,
        "azimuth": 0.0,
        "side": "ad",
        "expected": True
    },
    {
        "name": "Net Fault",
        "speed": 200.0,
        "angle": -8.725886,
        "azimuth": 0.0,
        "side": "deuce",
        "expected": False
    },
    {
        "name": "Long",
        "speed": 200.0,
        "angle": -7.102987,
        "azimuth": 0.0,
        "side": "deuce",
        "expected": False
    },
    {
        "name": "Wide",
        "speed": 200.0,
        "angle": -7.914436,
        "azimuth": 20.0,
        "side": "deuce",
        "expected": False
    }
]


print("=" * 60)
print("V5 — CLASSIFIER VERIFICATION")
print("=" * 60)

all_pass = True

for case in test_cases:

    result = classify_serve(
        speed_kmh=case["speed"],
        launch_angle_deg=case["angle"],
        azimuth_deg=case["azimuth"],
        omega=np.zeros(3),
        target_side=case["side"]
    )

    classification = (
        "LEGAL"
        if result["legal"]
        else "FAULT"
    )

    test_pass = (
        result["legal"]
        == case["expected"]
    )

    all_pass = all_pass and test_pass

    print("\n" + "-" * 60)
    print(case["name"])

    print(
        f"  θ = {case['angle']:.6f}°"
    )

    print(
        f"  φ = {case['azimuth']:.3f}°"
    )

    print(
        f"  Target = {case['side']}"
    )

    print(
        f"  Net clearance = "
        f"{result['net_clearance']:.4f} m"
    )

    print(
        f"  Landing x = "
        f"{result['landing_x']:.4f} m"
    )

    print(
        f"  Landing y = "
        f"{result['landing_y']:.4f} m"
    )

    print(
        f"  Clears net = "
        f"{result['clears_net']}"
    )

    print(
        f"  Inside depth = "
        f"{result['inside_depth']}"
    )

    print(
        f"  Inside width = "
        f"{result['inside_width']}"
    )

    print(
        f"  Classification = "
        f"{classification}"
    )

    print(
        f"  Test = "
        f"{'PASS' if test_pass else 'FAIL'}"
    )


print("\n" + "=" * 60)

if all_pass:
    print("V5 CLASSIFIER VERIFICATION: PASS")
else:
    print("V5 CLASSIFIER VERIFICATION: FAIL")

In [ ]:
# V5 Cell 6 — Build 2D Launch-Condition Grid

import numpy as np
import pandas as pd

# Fixed serve conditions
SPEED_GRID_KMH = 200.0
TARGET_SIDE = "deuce"
OMEGA_GRID = np.zeros(3)

# Launch-condition ranges
ANGLE_VALUES = np.arange(-10.0, -4.99, 0.5)
AZIMUTH_VALUES = np.arange(-20.0, 20.01, 0.5)

# Storage for results
grid_results = []

total_cases = len(ANGLE_VALUES) * len(AZIMUTH_VALUES)

print("=" * 60)
print("V5 — 2D ADMISSIBILITY GRID")
print("=" * 60)

print(f"Speed:             {SPEED_GRID_KMH:.1f} km/h")
print(f"Target side:       {TARGET_SIDE}")
print(f"Spin:              0 rpm")
print(f"Angle range:       {ANGLE_VALUES[0]:.1f}° to {ANGLE_VALUES[-1]:.1f}°")
print(f"Azimuth range:     {AZIMUTH_VALUES[0]:.1f}° to {AZIMUTH_VALUES[-1]:.1f}°")
print(f"Angle resolution:  0.5°")
print(f"Azimuth resolution: 0.5°")
print(f"Total trajectories: {total_cases}")

print("\nRunning grid...")

for angle in ANGLE_VALUES:

    for azimuth in AZIMUTH_VALUES:

        result = classify_serve(
            speed_kmh=SPEED_GRID_KMH,
            launch_angle_deg=angle,
            azimuth_deg=azimuth,
            omega=OMEGA_GRID,
            target_side=TARGET_SIDE
        )

        grid_results.append({
            "angle_deg": angle,
            "azimuth_deg": azimuth,
            "legal": result["legal"],
            "net_clearance_m": result["net_clearance"],
            "landing_x_m": result["landing_x"],
            "landing_y_m": result["landing_y"],
            "clears_net": result["clears_net"],
            "inside_depth": result["inside_depth"],
            "inside_width": result["inside_width"]
        })

print("Grid complete.")

# Convert results to a DataFrame
grid_df = pd.DataFrame(grid_results)

# Basic statistics
legal_count = int(grid_df["legal"].sum())
illegal_count = len(grid_df) - legal_count

print("\n" + "-" * 60)
print("GRID SUMMARY")
print("-" * 60)

print(f"Total cases:       {len(grid_df)}")
print(f"Legal cases:       {legal_count}")
print(f"Illegal cases:     {illegal_count}")
print(f"Legal fraction:    {legal_count / len(grid_df):.4f}")

print("\nGrid columns:")
print(list(grid_df.columns))

print("\nV5 Cell 6: GRID GENERATION COMPLETE")

In [ ]:
# V5 Cell 7 — Plot the 2D Admissibility Envelope

import matplotlib.pyplot as plt

# Reshape legal/illegal results into a 2D grid
legal_grid = grid_df.pivot(
    index="angle_deg",
    columns="azimuth_deg",
    values="legal"
)

plt.figure(figsize=(10, 6))

plt.imshow(
    legal_grid.values,
    origin="lower",
    aspect="auto",
    extent=[
        AZIMUTH_VALUES.min(),
        AZIMUTH_VALUES.max(),
        ANGLE_VALUES.min(),
        ANGLE_VALUES.max()
    ],
    interpolation="nearest"
)

plt.colorbar(
    label="Legal serve (0 = Fault, 1 = Legal)"
)

plt.xlabel("Launch azimuth (degrees)")
plt.ylabel("Launch angle (degrees)")
plt.title(
    "2D Serve Admissibility Envelope\n"
    "200 km/h, 3.0 m contact height, zero spin"
)

plt.tight_layout()
plt.show()

In [ ]:
# V5 Cell 8 — Extract Approximate Admissibility Boundaries

boundary_rows = []

for azimuth in AZIMUTH_VALUES:

    subset = grid_df[
        grid_df["azimuth_deg"] == azimuth
    ].sort_values("angle_deg")

    legal_subset = subset[
        subset["legal"] == True
    ]

    if len(legal_subset) > 0:

        min_angle = legal_subset["angle_deg"].min()
        max_angle = legal_subset["angle_deg"].max()
        angle_width = max_angle - min_angle

        boundary_rows.append({
            "azimuth_deg": azimuth,
            "min_legal_angle_deg": min_angle,
            "max_legal_angle_deg": max_angle,
            "angle_width_deg": angle_width
        })

boundary_df = pd.DataFrame(boundary_rows)

print("=" * 60)
print("V5 — APPROXIMATE ADMISSIBILITY BOUNDARIES")
print("=" * 60)

print(f"Azimuth values with legal serves: {len(boundary_df)}")
print(
    f"Azimuth range with legal serves: "
    f"{boundary_df['azimuth_deg'].min():.1f}° "
    f"to {boundary_df['azimuth_deg'].max():.1f}°"
)

print("\nMaximum angular width:")

max_width_row = boundary_df.loc[
    boundary_df["angle_width_deg"].idxmax()
]

print(
    f"  Azimuth = "
    f"{max_width_row['azimuth_deg']:.1f}°"
)

print(
    f"  Minimum angle = "
    f"{max_width_row['min_legal_angle_deg']:.1f}°"
)

print(
    f"  Maximum angle = "
    f"{max_width_row['max_legal_angle_deg']:.1f}°"
)

print(
    f"  Width = "
    f"{max_width_row['angle_width_deg']:.1f}°"
)

print("\nBoundary table:")
print(boundary_df.to_string(index=False))

print("\nV5 Cell 8: BOUNDARY EXTRACTION COMPLETE")

In [ ]:
# V5 Cell 9 — Continuous Boundary Detection

from scipy.optimize import brentq

def net_clearance_for_angle(angle_deg, azimuth_deg):
    """
    Return net clearance for a given launch angle and azimuth.
    Boundary occurs when clearance = 0.
    """
    result = classify_serve(
        speed_kmh=SPEED_GRID_KMH,
        launch_angle_deg=angle_deg,
        azimuth_deg=azimuth_deg,
        omega=OMEGA_GRID,
        target_side=TARGET_SIDE
    )

    return result["net_clearance"]


def landing_x_for_angle(angle_deg, azimuth_deg):
    """
    Return landing x-coordinate for a given launch angle and azimuth.
    Service-line boundary occurs when landing_x = SERVICE_LINE_X.
    """
    result = classify_serve(
        speed_kmh=SPEED_GRID_KMH,
        launch_angle_deg=angle_deg,
        azimuth_deg=azimuth_deg,
        omega=OMEGA_GRID,
        target_side=TARGET_SIDE
    )

    # Classifier returns "landing_x"
    return result["landing_x"]


# Azimuths for continuous boundary tracing
BOUNDARY_AZIMUTHS = np.arange(0.0, 13.01, 0.5)

boundary_results = []

for azimuth in BOUNDARY_AZIMUTHS:

    # --------------------------------------------------
    # Candidate launch angles
    # --------------------------------------------------

    net_angles = np.linspace(-10.0, -5.0, 21)

    # --------------------------------------------------
    # 1. Net boundary
    # --------------------------------------------------

    net_values = []

    for angle in net_angles:
        net_values.append(
            net_clearance_for_angle(angle, azimuth)
        )

    net_root = None

    for i in range(len(net_angles) - 1):

        if net_values[i] * net_values[i + 1] < 0:

            net_root = brentq(
                lambda a: net_clearance_for_angle(
                    a, azimuth
                ),
                net_angles[i],
                net_angles[i + 1]
            )

            break

    # --------------------------------------------------
    # 2. Service-line boundary
    # --------------------------------------------------

    landing_values = []

    for angle in net_angles:
        landing_values.append(
            landing_x_for_angle(angle, azimuth)
            - SERVICE_LINE_X
        )

    service_root = None

    for i in range(len(net_angles) - 1):

        if landing_values[i] * landing_values[i + 1] < 0:

            service_root = brentq(
                lambda a: (
                    landing_x_for_angle(a, azimuth)
                    - SERVICE_LINE_X
                ),
                net_angles[i],
                net_angles[i + 1]
            )

            break

    boundary_results.append({
        "azimuth_deg": azimuth,
        "net_boundary_deg": net_root,
        "service_boundary_deg": service_root
    })


continuous_boundary_df = pd.DataFrame(boundary_results)

print("=" * 60)
print("V5 — CONTINUOUS BOUNDARY DETECTION")
print("=" * 60)

print(
    continuous_boundary_df.to_string(index=False)
)

print("\nV5 Cell 9: CONTINUOUS BOUNDARY DETECTION COMPLETE")

In [ ]:
# V5 Cell 10 — Analyze Continuous Envelope Width

continuous_boundary_df["angle_width_deg"] = (
    continuous_boundary_df["service_boundary_deg"]
    - continuous_boundary_df["net_boundary_deg"]
)

print("=" * 60)
print("V5 — CONTINUOUS ENVELOPE ANALYSIS")
print("=" * 60)

# Maximum width
max_row = continuous_boundary_df.loc[
    continuous_boundary_df["angle_width_deg"].idxmax()
]

# Minimum width in the traced range
min_row = continuous_boundary_df.loc[
    continuous_boundary_df["angle_width_deg"].idxmin()
]

print(
    f"Maximum angular width: "
    f"{max_row['angle_width_deg']:.6f}°"
)

print(
    f"  at azimuth = "
    f"{max_row['azimuth_deg']:.2f}°"
)

print(
    f"Minimum angular width in traced range: "
    f"{min_row['angle_width_deg']:.6f}°"
)

print(
    f"  at azimuth = "
    f"{min_row['azimuth_deg']:.2f}°"
)

print("\n" + "-" * 60)
print("BOUNDARY TABLE")
print("-" * 60)

print(
    continuous_boundary_df[
        [
            "azimuth_deg",
            "net_boundary_deg",
            "service_boundary_deg",
            "angle_width_deg"
        ]
    ].to_string(index=False)
)

# --------------------------------------------------
# Plot 1 — Boundary curves
# --------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    continuous_boundary_df["azimuth_deg"],
    continuous_boundary_df["net_boundary_deg"],
    marker="o",
    label="Net boundary"
)

plt.plot(
    continuous_boundary_df["azimuth_deg"],
    continuous_boundary_df["service_boundary_deg"],
    marker="o",
    label="Service-line boundary"
)

plt.xlabel("Launch azimuth (degrees)")
plt.ylabel("Launch angle (degrees)")
plt.title(
    "Continuous Serve-Admissibility Boundaries\n"
    "200 km/h, zero spin"
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# --------------------------------------------------
# Plot 2 — Angular width
# --------------------------------------------------

plt.figure(figsize=(10, 5))

plt.plot(
    continuous_boundary_df["azimuth_deg"],
    continuous_boundary_df["angle_width_deg"],
    marker="o"
)

plt.xlabel("Launch azimuth (degrees)")
plt.ylabel("Admissible launch-angle width (degrees)")
plt.title(
    "Admissible Launch-Angle Width vs Azimuth\n"
    "200 km/h, zero spin"
)

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nV5 Cell 10: CONTINUOUS ENVELOPE ANALYSIS COMPLETE")

In [ ]:
# V5 Cell 11 — Find the Maximum Admissible Azimuth

from scipy.optimize import brentq

def boundary_width_at_azimuth(azimuth_deg):
    """
    Compute the continuous admissible launch-angle width
    at a specified azimuth.

    Width = service-line boundary - net boundary.
    """

    net_angles = np.linspace(-10.0, -5.0, 21)

    # --------------------------------------------------
    # Net boundary
    # --------------------------------------------------

    net_values = [
        net_clearance_for_angle(angle, azimuth_deg)
        for angle in net_angles
    ]

    net_root = None

    for i in range(len(net_angles) - 1):

        if net_values[i] * net_values[i + 1] < 0:

            net_root = brentq(
                lambda a: net_clearance_for_angle(
                    a, azimuth_deg
                ),
                net_angles[i],
                net_angles[i + 1]
            )

            break

    # --------------------------------------------------
    # Service-line boundary
    # --------------------------------------------------

    service_values = [
        landing_x_for_angle(angle, azimuth_deg)
        - SERVICE_LINE_X
        for angle in net_angles
    ]

    service_root = None

    for i in range(len(net_angles) - 1):

        if service_values[i] * service_values[i + 1] < 0:

            service_root = brentq(
                lambda a: (
                    landing_x_for_angle(
                        a, azimuth_deg
                    )
                    - SERVICE_LINE_X
                ),
                net_angles[i],
                net_angles[i + 1]
            )

            break

    if net_root is None or service_root is None:
        return np.nan

    return service_root - net_root


# --------------------------------------------------
# First extend the search
# --------------------------------------------------

extended_azimuths = np.arange(13.0, 21.0, 1.0)

extended_widths = []

for azimuth in extended_azimuths:

    width = boundary_width_at_azimuth(azimuth)

    extended_widths.append({
        "azimuth_deg": azimuth,
        "angle_width_deg": width
    })


extended_width_df = pd.DataFrame(extended_widths)

print("=" * 60)
print("V5 — EXTENDED AZIMUTH SEARCH")
print("=" * 60)

print(
    extended_width_df.to_string(index=False)
)

print("\nV5 Cell 11: EXTENDED AZIMUTH SEARCH COMPLETE")

In [ ]:
# V5 Cell 12 — Identify the True Lateral Boundary

# Fine launch-angle scan
FINE_ANGLES = np.arange(-10.0, -4.99, 0.1)

# Extended azimuth range
CHECK_AZIMUTHS = np.arange(13.0, 20.01, 1.0)

lateral_results = []

print("=" * 60)
print("V5 — TRUE LATERAL BOUNDARY SEARCH")
print("=" * 60)

for azimuth in CHECK_AZIMUTHS:

    legal_angles = []

    for angle in FINE_ANGLES:

        result = classify_serve(
            speed_kmh=SPEED_GRID_KMH,
            launch_angle_deg=angle,
            azimuth_deg=azimuth,
            omega=OMEGA_GRID,
            target_side=TARGET_SIDE
        )

        if result["legal"]:
            legal_angles.append(angle)

    if len(legal_angles) > 0:

        lateral_results.append({
            "azimuth_deg": azimuth,
            "min_legal_angle_deg": min(legal_angles),
            "max_legal_angle_deg": max(legal_angles),
            "legal_angle_width_deg":
                max(legal_angles) - min(legal_angles),
            "legal_exists": True
        })

    else:

        lateral_results.append({
            "azimuth_deg": azimuth,
            "min_legal_angle_deg": np.nan,
            "max_legal_angle_deg": np.nan,
            "legal_angle_width_deg": 0.0,
            "legal_exists": False
        })


lateral_boundary_df = pd.DataFrame(lateral_results)

print("\n" + "-" * 60)
print("LEGAL REGION AFTER ALL CONSTRAINTS")
print("-" * 60)

print(
    lateral_boundary_df.to_string(index=False)
)

# Determine last azimuth with at least one legal serve
legal_azimuths = lateral_boundary_df[
    lateral_boundary_df["legal_exists"]
]["azimuth_deg"]

if len(legal_azimuths) > 0:

    last_legal_azimuth = legal_azimuths.max()

    print("\n" + "-" * 60)
    print("SUMMARY")
    print("-" * 60)

    print(
        f"Last sampled azimuth with a legal serve: "
        f"{last_legal_azimuth:.1f}°"
    )

else:

    print("\nNo legal serves found in the tested range.")

print("\nV5 Cell 12: TRUE LATERAL BOUNDARY SEARCH COMPLETE")

In [ ]:
# V5 Cell 13 — Refine the Lateral Boundary

FINE_AZIMUTHS = np.arange(13.0, 14.001, 0.1)
FINE_ANGLES_LATERAL = np.arange(-8.5, -6.8, 0.05)

refined_lateral_results = []

print("=" * 60)
print("V5 — REFINED LATERAL BOUNDARY SEARCH")
print("=" * 60)

for azimuth in FINE_AZIMUTHS:

    legal_angles = []
    max_landing_y = -np.inf
    best_angle = np.nan
    best_landing_x = np.nan
    best_landing_y = np.nan

    for angle in FINE_ANGLES_LATERAL:

        result = classify_serve(
            speed_kmh=SPEED_GRID_KMH,
            launch_angle_deg=angle,
            azimuth_deg=azimuth,
            omega=OMEGA_GRID,
            target_side=TARGET_SIDE
        )

        # Track the trajectory that reaches farthest laterally
        if result["landing_y"] > max_landing_y:
            max_landing_y = result["landing_y"]
            best_angle = angle
            best_landing_x = result["landing_x"]
            best_landing_y = result["landing_y"]

        if result["legal"]:
            legal_angles.append(angle)

    if len(legal_angles) > 0:

        refined_lateral_results.append({
            "azimuth_deg": azimuth,
            "min_legal_angle_deg": min(legal_angles),
            "max_legal_angle_deg": max(legal_angles),
            "legal_exists": True,
            "max_landing_y_m": max_landing_y,
            "angle_at_max_y_deg": best_angle,
            "landing_x_at_max_y_m": best_landing_x
        })

    else:

        refined_lateral_results.append({
            "azimuth_deg": azimuth,
            "min_legal_angle_deg": np.nan,
            "max_legal_angle_deg": np.nan,
            "legal_exists": False,
            "max_landing_y_m": max_landing_y,
            "angle_at_max_y_deg": best_angle,
            "landing_x_at_max_y_m": best_landing_x
        })


refined_lateral_df = pd.DataFrame(
    refined_lateral_results
)

print("\n" + "-" * 60)
print("REFINED RESULTS")
print("-" * 60)

print(
    refined_lateral_df.to_string(index=False)
)

# Find the last azimuth with a legal trajectory
legal_rows = refined_lateral_df[
    refined_lateral_df["legal_exists"]
]

if len(legal_rows) > 0:

    last_legal = legal_rows.iloc[-1]

    print("\n" + "-" * 60)
    print("REFINED BOUNDARY ESTIMATE")
    print("-" * 60)

    print(
        f"Last sampled legal azimuth: "
        f"{last_legal['azimuth_deg']:.1f}°"
    )

    print(
        f"Legal angle range there: "
        f"{last_legal['min_legal_angle_deg']:.2f}° "
        f"to "
        f"{last_legal['max_legal_angle_deg']:.2f}°"
    )

    print(
        f"Maximum landing y in scan: "
        f"{last_legal['max_landing_y_m']:.4f} m"
    )

    print(
        f"Service-box boundary: "
        f"{SERVICE_BOX_WIDTH:.4f} m"
    )

print("\nV5 Cell 13: REFINED LATERAL BOUNDARY SEARCH COMPLETE")

In [ ]:
# V5 Cell 14 — Continuous Corner-Boundary Solution

from scipy.optimize import root

def corner_boundary_residuals(variables):
    """
    Solve for the launch angle and azimuth at which
    the serve lands exactly on the outer service-box corner.

    Equations:
        landing_x = SERVICE_LINE_X
        landing_y = SERVICE_BOX_WIDTH
    """

    angle_deg, azimuth_deg = variables

    result = classify_serve(
        speed_kmh=SPEED_GRID_KMH,
        launch_angle_deg=angle_deg,
        azimuth_deg=azimuth_deg,
        omega=OMEGA_GRID,
        target_side=TARGET_SIDE
    )

    return np.array([
        result["landing_x"] - SERVICE_LINE_X,
        result["landing_y"] - SERVICE_BOX_WIDTH
    ])


# Initial guess from the refined grid
initial_guess = np.array([
    -7.80,   # launch angle
    13.75    # azimuth
])

solution_corner = root(
    corner_boundary_residuals,
    initial_guess,
    method="hybr"
)

print("=" * 60)
print("V5 — CONTINUOUS CORNER-BOUNDARY SOLUTION")
print("=" * 60)

print(f"Solver success: {solution_corner.success}")
print(f"Solver message: {solution_corner.message}")

if solution_corner.success:

    corner_angle = solution_corner.x[0]
    corner_azimuth = solution_corner.x[1]

    corner_result = classify_serve(
        speed_kmh=SPEED_GRID_KMH,
        launch_angle_deg=corner_angle,
        azimuth_deg=corner_azimuth,
        omega=OMEGA_GRID,
        target_side=TARGET_SIDE
    )

    print("\n" + "-" * 60)
    print("CORNER BOUNDARY")
    print("-" * 60)

    print(
        f"Launch angle:       {corner_angle:.8f}°"
    )

    print(
        f"Launch azimuth:     {corner_azimuth:.8f}°"
    )

    print(
        f"Landing x:          "
        f"{corner_result['landing_x']:.8f} m"
    )

    print(
        f"Landing y:          "
        f"{corner_result['landing_y']:.8f} m"
    )

    print(
        f"Net clearance:      "
        f"{corner_result['net_clearance']:.8f} m"
    )

    print(
        f"Clears net:         "
        f"{corner_result['clears_net']}"
    )

    print("\nResiduals:")

    residuals = corner_boundary_residuals(
        solution_corner.x
    )

    print(
        f"  x residual: "
        f"{residuals[0]:.3e} m"
    )

    print(
        f"  y residual: "
        f"{residuals[1]:.3e} m"
    )

    print("\nV5 Cell 14: CORNER-BOUNDARY SOLUTION COMPLETE")

else:

    print("\nCorner-boundary solver did not converge.")

    print("\nV5 Cell 14: SOLVER FAILED")

In [ ]:
# V5 Cell 15 — True Admissibility-Envelope Endpoint

from scipy.optimize import root


def envelope_endpoint_residuals(variables):
    """
    Solve for the launch angle and azimuth where
    the net constraint and lateral service-box constraint
    become simultaneously active.

    Equations:

        net clearance = 0
        landing y = SERVICE_BOX_WIDTH
    """

    angle_deg, azimuth_deg = variables

    result = classify_serve(
        speed_kmh=SPEED_GRID_KMH,
        launch_angle_deg=angle_deg,
        azimuth_deg=azimuth_deg,
        omega=OMEGA_GRID,
        target_side=TARGET_SIDE
    )

    return np.array([
        result["net_clearance"],
        result["landing_y"] - SERVICE_BOX_WIDTH
    ])


# Initial guess based on Cell 13:
# near the transition between 13.7° and 13.8°
initial_guess_endpoint = np.array([
    -7.82,
    13.75
])


endpoint_solution = root(
    envelope_endpoint_residuals,
    initial_guess_endpoint,
    method="hybr"
)


print("=" * 60)
print("V5 — TRUE ADMISSIBILITY-ENVELOPE ENDPOINT")
print("=" * 60)

print(
    f"Solver success: "
    f"{endpoint_solution.success}"
)

print(
    f"Solver message: "
    f"{endpoint_solution.message}"
)


if endpoint_solution.success:

    endpoint_angle = endpoint_solution.x[0]
    endpoint_azimuth = endpoint_solution.x[1]

    endpoint_result = classify_serve(
        speed_kmh=SPEED_GRID_KMH,
        launch_angle_deg=endpoint_angle,
        azimuth_deg=endpoint_azimuth,
        omega=OMEGA_GRID,
        target_side=TARGET_SIDE
    )

    print("\n" + "-" * 60)
    print("ENVELOPE ENDPOINT")
    print("-" * 60)

    print(
        f"Launch angle:       "
        f"{endpoint_angle:.8f}°"
    )

    print(
        f"Launch azimuth:     "
        f"{endpoint_azimuth:.8f}°"
    )

    print(
        f"Net clearance:      "
        f"{endpoint_result['net_clearance']:.10f} m"
    )

    print(
        f"Landing x:          "
        f"{endpoint_result['landing_x']:.8f} m"
    )

    print(
        f"Landing y:          "
        f"{endpoint_result['landing_y']:.8f} m"
    )

    print(
        f"Inside depth:       "
        f"{endpoint_result['inside_depth']}"
    )

    print(
        f"Inside width:       "
        f"{endpoint_result['inside_width']}"
    )

    print("\nResiduals:")

    residuals = envelope_endpoint_residuals(
        endpoint_solution.x
    )

    print(
        f"  Net-clearance residual: "
        f"{residuals[0]:.3e} m"
    )

    print(
        f"  Lateral residual:       "
        f"{residuals[1]:.3e} m"
    )

    print("\nV5 Cell 15: TRUE ENDPOINT SOLUTION COMPLETE")

else:

    print(
        "\nTrue envelope endpoint solver "
        "did not converge."
    )

    print("\nV5 Cell 15: SOLVER FAILED")

In [ ]:
# V5 Cell 16 — Verify the Continuous Envelope Endpoint

print("=" * 60)
print("V5 — ENVELOPE ENDPOINT VERIFICATION")
print("=" * 60)

test_points = [
    {
        "name": "Inside endpoint — lower azimuth",
        "angle": endpoint_angle,
        "azimuth": endpoint_azimuth - 0.05
    },
    {
        "name": "Endpoint",
        "angle": endpoint_angle,
        "azimuth": endpoint_azimuth
    },
    {
        "name": "Outside endpoint — higher azimuth",
        "angle": endpoint_angle,
        "azimuth": endpoint_azimuth + 0.05
    }
]

for test in test_points:

    result = classify_serve(
        speed_kmh=SPEED_GRID_KMH,
        launch_angle_deg=test["angle"],
        azimuth_deg=test["azimuth"],
        omega=OMEGA_GRID,
        target_side=TARGET_SIDE
    )

    print("\n" + "-" * 60)
    print(test["name"])

    print(
        f"  Angle:          "
        f"{test['angle']:.8f}°"
    )

    print(
        f"  Azimuth:        "
        f"{test['azimuth']:.8f}°"
    )

    print(
        f"  Net clearance:  "
        f"{result['net_clearance']:.8f} m"
    )

    print(
        f"  Landing x:      "
        f"{result['landing_x']:.8f} m"
    )

    print(
        f"  Landing y:      "
        f"{result['landing_y']:.8f} m"
    )

    print(
        f"  Clears net:     "
        f"{result['clears_net']}"
    )

    print(
        f"  Inside depth:   "
        f"{result['inside_depth']}"
    )

    print(
        f"  Inside width:   "
        f"{result['inside_width']}"
    )

    print(
        f"  Legal:          "
        f"{result['legal']}"
    )

print("\n" + "=" * 60)
print("V5 Cell 16: ENDPOINT VERIFICATION COMPLETE")
print("=" * 60)